GLOBAL IMPORTS AND CONFIGS

In [1]:
from __future__ import annotations

import json
import math
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'app').exists():
    PROJECT_ROOT = Path.cwd().resolve().parent

sys.path.insert(0, str(PROJECT_ROOT))

from app.api.dependencies import get_llm_service, get_rag_pipeline, get_vector_store
from app.core.config import get_settings
from app.core.logging import configure_logging
from app.rag.retriever import HybridRetriever
from app.services.embedding_service import get_embedding_service

settings = get_settings()
configure_logging('WARNING')

DATA_PATH = Path(settings.data_path)
TOP_K = 10
NUM_EXACT_CASES = 40
NUM_CATEGORY_CASES = 20
RANDOM_SEED = 42
RUN_GENERATION_EVAL = False
GENERATION_CASE_LIMIT = 5
VECTOR_SAMPLE_SIZE = 2500
FOCUS_QUERY = 'remote python developer jobs with machine learning experience'

print(f'Project root: {PROJECT_ROOT}')
print(f'Data path: {DATA_PATH}')


C:\Users\mahar\miniconda3\envs\LeapFrog_Ass2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: C:\Users\mahar\PycharmProjects\PythonProject1
Data path: C:\Users\mahar\PycharmProjects\PythonProject1\data\LFJobs.csv


LOADING SERVICES AND DATASET

In [2]:
embedding_service = get_embedding_service()
vector_store = get_vector_store()
retriever = HybridRetriever(vector_store=vector_store, embedding_service=embedding_service)

df = pd.read_csv(DATA_PATH, dtype=str).fillna('')

print(f'Rows in source CSV: {len(df):,}')
print(f'Vector chunks in Chroma: {vector_store.document_count:,}')
print(f'Embedding model: {embedding_service.model_name}')
print(f'Vector store ready: {vector_store.is_ready()}')

df.head(3)

Rows in source CSV: 1,000
Vector chunks in Chroma: 16,501
Embedding model: BAAI/bge-small-en-v1.5
Vector store ready: True


,ID,Job Category,Job Title,Company Name,Publication Date,Job Location,Job Level,Tags,Job Description
0,LF0001,Data and Analytics,"DIR, Equities Quant",Merrill,2025-07-28T23:00:54Z,"New York, NY",Mid Level,,<p><b>Job Description:</b><br><br>At Bank of A...
1,LF0002,Data and Analytics,Lead Administrator - L1,Wipro,2025-06-15T23:43:28Z,"Hyderabad, India",Senior Level,,"<p>Wipro Limited (NYSE: WIT, BSE: 507685, NSE:..."
2,LF0003,Data and Analytics,Principal DEX Solutions Architect,Eaton,2025-07-30T11:38:27Z,"Chicago, IL",Senior Level,,<p>Eaton's Digital Employee Experience (DEX) t...


PSEUDO BENCHMARK CONSTRUCTION AND LABELING

In [3]:
@dataclass(frozen=True)
class EvalCase:
    case_id: str
    query: str
    relevant_job_ids: set[str]
    case_type: str
    expected_terms: tuple[str, ...] = ()
    filters: dict[str, Any] | None = None


def _clean(value: str) -> str:
    return str(value).strip()


def build_eval_cases(data: pd.DataFrame) -> list[EvalCase]:
    rng = np.random.default_rng(RANDOM_SEED)
    valid = data[
        (data['ID'].str.len() > 0)
        & (data['Job Title'].str.len() > 0)
        & (data['Company Name'].str.len() > 0)
        & (data['Job Description'].str.len() > 80)
    ].copy()

    exact_sample = valid.sample(
        n=min(NUM_EXACT_CASES, len(valid)),
        random_state=RANDOM_SEED,
    )

    cases: list[EvalCase] = []
    for i, row in exact_sample.reset_index(drop=True).iterrows():
        title = _clean(row['Job Title'])
        company = _clean(row['Company Name'])
        location = _clean(row.get('Job Location', ''))
        cases.append(
            EvalCase(
                case_id=f'exact_{i:03d}',
                query=f'What are the requirements and responsibilities for the {title} role at {company} in {location}?',
                relevant_job_ids={_clean(row['ID'])},
                case_type='exact_job',
                expected_terms=tuple(t.lower() for t in [title, company] if t),
            )
        )

    grouped = (
        valid.groupby(['Job Category', 'Job Location'])['ID']
        .apply(lambda ids: sorted(set(str(x) for x in ids if str(x))))
        .reset_index(name='relevant_ids')
    )
    grouped['num_relevant'] = grouped['relevant_ids'].str.len()
    grouped = grouped[grouped['num_relevant'].between(3, 80)]

    if len(grouped):
        broad_sample = grouped.sample(
            n=min(NUM_CATEGORY_CASES, len(grouped)),
            random_state=RANDOM_SEED + 1,
        )
        for i, row in broad_sample.reset_index(drop=True).iterrows():
            category = _clean(row['Job Category'])
            location = _clean(row['Job Location'])
            cases.append(
                EvalCase(
                    case_id=f'broad_{i:03d}',
                    query=f'Find relevant {category} jobs in {location}.',
                    relevant_job_ids=set(row['relevant_ids']),
                    case_type='category_location',
                    expected_terms=tuple(t.lower() for t in [category, location] if t),
                )
            )

    return cases


eval_cases = build_eval_cases(df)
case_preview = pd.DataFrame([
    {
        'case_id': c.case_id,
        'case_type': c.case_type,
        'query': c.query,
        'num_relevant_ids': len(c.relevant_job_ids),
    }
    for c in eval_cases
])

print(f'Evaluation cases: {len(eval_cases)}')
case_preview.head(10)

Evaluation cases: 60


,case_id,case_type,query,num_relevant_ids
0,exact_000,exact_job,What are the requirements and responsibilities...,1
1,exact_001,exact_job,What are the requirements and responsibilities...,1
2,exact_002,exact_job,What are the requirements and responsibilities...,1
3,exact_003,exact_job,What are the requirements and responsibilities...,1
4,exact_004,exact_job,What are the requirements and responsibilities...,1
5,exact_005,exact_job,What are the requirements and responsibilities...,1
6,exact_006,exact_job,What are the requirements and responsibilities...,1
7,exact_007,exact_job,What are the requirements and responsibilities...,1
8,exact_008,exact_job,What are the requirements and responsibilities...,1
9,exact_009,exact_job,What are the requirements and responsibilities...,1


RETRIEVAL EVALUATION

In [8]:
def evaluate_retrieval_case(case: EvalCase, top_k: int = TOP_K) -> dict[str, Any]:
    started = time.perf_counter()
    results = retriever.retrieve(query=case.query, top_k=top_k, filters=case.filters)
    latency_ms = (time.perf_counter() - started) * 1000

    retrieved_job_ids = [str(r.get('metadata', {}).get('job_id', '')) for r in results]
    relevant_flags = [job_id in case.relevant_job_ids for job_id in retrieved_job_ids]

    first_relevant_rank = next((i + 1 for i, flag in enumerate(relevant_flags) if flag), None)
    hit = first_relevant_rank is not None
    precision = sum(relevant_flags) / max(len(results), 1)
    mrr = 1 / first_relevant_rank if first_relevant_rank else 0.0

    expected_terms = [term for term in case.expected_terms if term]
    if expected_terms:
        matching_chunks = 0
        for result in results:
            text = ' '.join([
                str(result.get('text', '')),
                json.dumps(result.get('metadata', {}), ensure_ascii=False),
            ]).lower()
            if any(term in text for term in expected_terms):
                matching_chunks += 1
        term_overlap = matching_chunks / max(len(results), 1)
    else:
        term_overlap = np.nan

    similarities = [float(r.get('similarity_score', 0.0)) for r in results]

    return {
        'case_id': case.case_id,
        'case_type': case.case_type,
        'query': case.query,
        'num_relevant_ids': len(case.relevant_job_ids),
        'hit_at_k': hit,
        'first_relevant_rank': first_relevant_rank,
        'mrr': mrr,
        'precision_at_k': precision,
        'term_overlap_at_k': term_overlap,
        'latency_ms': latency_ms,
        'avg_similarity': float(np.mean(similarities)) if similarities else 0.0,
        'max_similarity': float(np.max(similarities)) if similarities else 0.0,
        'retrieved_job_ids': retrieved_job_ids,
        'top_result_title': results[0].get('metadata', {}).get('job_title', '') if results else '',
        'top_result_company': results[0].get('metadata', {}).get('company_name', '') if results else '',
    }


retrieval_rows = [evaluate_retrieval_case(case) for case in eval_cases]
retrieval_eval = pd.DataFrame(retrieval_rows)
summary = retrieval_eval.groupby('case_type').agg(
    cases=('case_id', 'count'),
    hit_at_k=('hit_at_k', 'mean'),
    mrr=('mrr', 'mean'),
    precision_at_k=('precision_at_k', 'mean'),
    term_overlap_at_k=('term_overlap_at_k', 'mean'),
    latency_ms_median=('latency_ms', 'median'),
    latency_ms_p95=('latency_ms', lambda x: np.percentile(x, 95)),
    avg_similarity=('avg_similarity', 'mean'),
).reset_index()

summary

,case_type,cases,hit_at_k,mrr,precision_at_k,term_overlap_at_k,latency_ms_median,latency_ms_p95,avg_similarity
0,category_location,20,1.0,0.862500,0.345,0.99,332.93585,371.737730,0.853306
1,exact_job,40,1.0,0.930833,0.270,0.86,318.48590,577.484505,0.874374


VECTOR SPACE SIMILARITY MAP

In [6]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

try:
    from IPython.display import HTML, display
    import plotly.express as px
    import plotly.io as pio
except ImportError as exc:
    raise ImportError(
        'Plotly is required for the interactive 3D plot. Install dev dependencies with: '
        'pip install -r requirements-dev.txt'
    ) from exc


def cosine_similarity_to_query(matrix: np.ndarray, query_vector: np.ndarray) -> np.ndarray:
    matrix_norm = np.linalg.norm(matrix, axis=1)
    query_norm = np.linalg.norm(query_vector)
    denom = np.maximum(matrix_norm * query_norm, 1e-12)
    return (matrix @ query_vector) / denom


sample_size = min(VECTOR_SAMPLE_SIZE, vector_store.document_count)
raw = vector_store._collection.get(
    limit=sample_size,
    include=['embeddings', 'documents', 'metadatas'],
)

embeddings = np.asarray(raw['embeddings'], dtype=np.float32)
metadatas = raw['metadatas']
documents = raw['documents']
ids = raw['ids']

query_embedding = np.asarray(embedding_service.embed_query(FOCUS_QUERY), dtype=np.float32)
query_similarity = cosine_similarity_to_query(embeddings, query_embedding)

scaler = StandardScaler()
scaled_embeddings = scaler.fit_transform(embeddings)
pca = PCA(n_components=3, random_state=RANDOM_SEED)
coords = pca.fit_transform(scaled_embeddings)
query_coords = pca.transform(scaler.transform(query_embedding.reshape(1, -1)))[0]

similarity_min = float(query_similarity.min())
similarity_range = float(query_similarity.max() - query_similarity.min()) or 1.0
marker_size = 3.0 + ((query_similarity - similarity_min) / similarity_range) * 7.0

plot_df = pd.DataFrame({
    'x': coords[:, 0],
    'y': coords[:, 1],
    'z': coords[:, 2],
    'chunk_id': ids,
    'job_id': [m.get('job_id', '') for m in metadatas],
    'job_title': [m.get('job_title', '') for m in metadatas],
    'company_name': [m.get('company_name', '') for m in metadatas],
    'job_category': [m.get('job_category', 'Unknown') or 'Unknown' for m in metadatas],
    'job_level': [m.get('job_level', 'Unknown') or 'Unknown' for m in metadatas],
    'job_location': [m.get('job_location', '') for m in metadatas],
    'query_similarity': query_similarity,
    'marker_size': marker_size,
    'point_type': 'job chunk',
    'text_preview': [(doc or '')[:220].replace('\n', ' ') for doc in documents],
})

query_row = pd.DataFrame([{
    'x': query_coords[0],
    'y': query_coords[1],
    'z': query_coords[2],
    'chunk_id': 'FOCUS_QUERY',
    'job_id': '',
    'job_title': FOCUS_QUERY,
    'company_name': 'Query vector',
    'job_category': 'Query',
    'job_level': 'Query',
    'job_location': '',
    'query_similarity': 1.0,
    'marker_size': 14.0,
    'point_type': 'focus query',
    'text_preview': FOCUS_QUERY,
}])
plot_with_query = pd.concat([plot_df, query_row], ignore_index=True)

fig = px.scatter_3d(
    plot_with_query,
    x='x',
    y='y',
    z='z',
    color='query_similarity',
    symbol='point_type',
    size='marker_size',
    color_continuous_scale='Viridis',
    hover_name='job_title',
    hover_data={
        'query_similarity': ':.3f',
        'company_name': True,
        'job_category': True,
        'job_level': True,
        'job_location': True,
        'job_id': True,
        'chunk_id': True,
        'text_preview': True,
        'marker_size': False,
        'point_type': False,
        'x': False,
        'y': False,
        'z': False,
    },
    title=f'3D Vector Similarity Map for Query: {FOCUS_QUERY}',
    opacity=0.82,
    height=820,
)

fig.update_traces(marker={'line': {'width': 0.25, 'color': 'rgba(15,23,42,0.25)'}})
fig.update_layout(
    template='plotly_white',
    coloraxis_colorbar={'title': 'Cosine similarity'},
    legend_title_text='Point type',
    margin={'l': 0, 'r': 0, 't': 70, 'b': 0},
    scene={
        'xaxis_title': 'PCA 1',
        'yaxis_title': 'PCA 2',
        'zaxis_title': 'PCA 3',
        'xaxis': {'backgroundcolor': 'rgb(248,250,252)', 'gridcolor': 'rgb(226,232,240)'},
        'yaxis': {'backgroundcolor': 'rgb(248,250,252)', 'gridcolor': 'rgb(226,232,240)'},
        'zaxis': {'backgroundcolor': 'rgb(248,250,252)', 'gridcolor': 'rgb(226,232,240)'},
    },
)

display(HTML(fig.to_html(include_plotlyjs='cdn', full_html=False)))

FINAL REPORTS

In [7]:
score_fig = px.histogram(
    retrieval_eval,
    x='max_similarity',
    color='case_type',
    marginal='box',
    nbins=30,
    title='Maximum Retrieved Similarity by Evaluation Case',
    labels={'max_similarity': 'Max top-k similarity'},
    template='plotly_white',
    height=520,
)
display(HTML(score_fig.to_html(include_plotlyjs='cdn', full_html=False)))